[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sunilmogadati/systems-in-production/blob/main/implementation/notebooks/Deep_Learning_CNN.ipynb) &nbsp; [![View on GitHub](https://img.shields.io/badge/GitHub-View%20Source-blue?logo=github)](https://github.com/sunilmogadati/systems-in-production/blob/main/implementation/notebooks/Deep_Learning_CNN.ipynb)
Playbook: [AI / Deep Learning](../../playbooks/ai/deep-learning/README.md)


# AI Engineer Accelerator: CNN Deep Dive**From Pixels to Predictions — How Convolutional Neural Networks See**| | ||:---|:---|| **Program** | [Emergtech AI Engineer Accelerator](https://github.com/sunilmogadati/systems-in-production) || **Author** | Sunil Mogadati || **Community** | [DeliveryMomentum](https://www.skool.com/deliverymomentum) || **Week** | 3-4 — Deep Learning: Computer Vision with CNNs || **Prerequisites** | [Deep Learning and PyTorch](https://colab.research.google.com/github/sunilmogadati/systems-in-production/blob/main/implementation/notebooks/Deep_Learning_PyTorch.ipynb) || **Pluralsight Companion** | Convolutional Neural Networks (CNNs), Image Classification with PyTorch || **Also serves as** | CSCI 5931 Deep Learning — Exam 2 Study Guide (CNN section) |

## What You Will Learn- Why fully connected networks fail for images — and what CNNs do differently- The two principles that make CNNs work: **local connectivity** and **parameter sharing**- How convolution, pooling, and flattening work — with math, code, and analogies- How to calculate output shapes and count parameters by hand (exam-critical)- What 1x1 convolutions are and why every modern architecture uses them- How to build, train, and debug CNNs in PyTorch and Keras- The evolution from LeNet (1998) to ResNet (2015) — what each architecture discovered- Real-world CNN applications: medical imaging, self-driving cars, manufacturing, OCR

## Why This MattersIn the previous notebook, you built a neural network that classifies MNIST digits by **flattening** the 28x28 image into a 784-dimensional vector. It works — 97%+ accuracy. So why change anything?Because flattening **destroys spatial structure**. The network treats pixel (0,0) and pixel (27,27) as equally unrelated. It has no concept that neighboring pixels form edges, edges form shapes, and shapes form objects. For simple digits, this is tolerable. For real images (faces, cars, X-rays), it's catastrophic.CNNs solve this by **preserving the 2D structure** of images and processing them through learnable filters that detect patterns locally. This is the architecture behind:- Your phone's face unlock (Face ID)- Tesla Autopilot's pedestrian detection- Instagram's filters and effects- Medical AI that spots tumors in X-rays- ChatGPT's ability to understand images (vision models use CNN backbones)If you're an AI Engineer, you **will** encounter CNNs — either building vision features, debugging a pretrained model, or making architectural decisions about when to use them.

## Key Terms| Abbreviation | Full Form | Plain English ||:---|:---|:---|| **CNN** | Convolutional Neural Network | Neural network that preserves spatial structure — scans for patterns like edges, shapes, textures || **Conv2d** | 2D Convolution Layer | A layer that slides filters across an image to detect patterns || **Filter / Kernel** | Feature Detector | A small matrix of learnable weights that detects one specific pattern (edge, curve, texture) || **Feature Map** | Activation Map | The output of applying one filter to an image — shows where and how strongly the pattern was detected || **Stride** | Step Size | How many pixels the filter moves between positions (stride=1: every pixel, stride=2: skip one) || **Padding** | Zero Border | Zeros added around the image border so the filter can cover edge pixels || **Pooling** | Downsampling | Shrinks feature maps by summarizing local regions (max or average) || **1x1 Conv** | Pointwise Convolution | A 1x1 filter that changes channel depth without touching spatial dimensions || **Receptive Field** | Field of View | The region of the original image that influences one output neuron || **Weight Sharing** | Parameter Reuse | Same filter weights used at every spatial position — the key efficiency trick of CNNs || **FC** | Fully Connected Layer | Traditional dense layer where every input connects to every output || **BN** | Batch Normalization | Normalizes layer outputs to stabilize training |

## Math Symbols Used in This Notebook| Symbol | Name | What It Means Here ||:---|:---|:---|| **N** | Input dimension | Height or width of the input image (e.g., 32 for a 32x32 image) || **F** | Filter size | Height/width of the convolution filter (e.g., 3 for a 3x3 filter) || **S** | Stride | How many pixels the filter moves per step || **P** | Padding | Number of zero-pixel rows/columns added around the image border || **K** | Number of filters | How many different filters in one convolution layer || **D** | Depth / Channels | Number of channels (3 for RGB, 1 for grayscale, or number of feature maps) || **O** | Output dimension | Height or width of the output feature map: O = (N + 2P - F) / S + 1 |

---

## Part Ib: Evaluation Metrics and Multi-Class Classification (Exam Formulas)

These topics were covered in Session 9 (Feb 19) — the first lecture in the Exam 2 scope. The professor wrote these formulas on the board explicitly.

## 4b. Confusion Matrix — The Diagnostic Dashboard

Given a binary classifier's predictions vs actual labels:

```
                      Predicted
                    0 (Neg)    1 (Pos)
Actual  0 (Neg)      TN         FP        ← All actual negatives
        1 (Pos)      FN         TP        ← All actual positives
                     ↑          ↑
              Predicted Neg  Predicted Pos
```

- **TP (True Positive):** Actually positive, predicted positive ✓
- **TN (True Negative):** Actually negative, predicted negative ✓
- **FP (False Positive):** Actually negative, predicted positive ✗ (false alarm)
- **FN (False Negative):** Actually positive, predicted negative ✗ (missed detection)

**Diagonal = correct predictions. Off-diagonal = errors.**

## 4c. Precision, Recall, F1 Score — The Formulas

**Precision** — "Of everything the model called positive, how many actually were?"

```
Precision = TP / (TP + FP)
```
Hurt by false positives (crying wolf). High precision = few false alarms.

**Recall** — "Of everything that actually was positive, how many did the model catch?"

```
Recall = TP / (TP + FN)
```
Hurt by false negatives (missed detections). High recall = catches almost everything.

**F1 Score** — Harmonic mean (if either is bad, F1 is bad):

```
F1 = 2 × Precision × Recall / (Precision + Recall)
```

**Accuracy** — Overall correctness:

```
Accuracy = (TP + TN) / (TP + TN + FP + FN)
```

### Worked Example

**Given confusion matrix:**
```
              Predicted
            0 (Neg)  1 (Pos)
Actual 0      80       5        (85 actual negatives)
Actual 1      10      55        (65 actual positives)
```

```
TP = 55, TN = 80, FP = 5, FN = 10
Total = 150

Precision = 55 / (55 + 5) = 55/60 = 0.917 (91.7%)
Recall    = 55 / (55 + 10) = 55/65 = 0.846 (84.6%)
F1        = 2 × 0.917 × 0.846 / (0.917 + 0.846) = 1.552 / 1.763 = 0.880 (88.0%)
Accuracy  = (55 + 80) / 150 = 135/150 = 0.900 (90.0%)
```

**Key insight from the professor:** "If one of precision or recall is bad, F1 gets pulled down." This is why F1 uses harmonic mean, not arithmetic mean. If precision=1.0 and recall=0.01, arithmetic mean=0.505 (looks OK), but F1=0.02 (correctly shows the model is terrible).

## 4d. Softmax — Multi-Class Output Activation

**Problem:** For multi-class classification (10 MNIST digits), sigmoid on each output neuron independently can produce conflicting results (multiple neurons claim "I'm the answer").

**Solution:** Softmax normalizes across ALL output neurons so they sum to 1:

```
softmax(z_i) = e^(z_i) / Σ e^(z_j)    for all j
```

### Worked Example

**Raw outputs (logits) from 3 output neurons:** z = [2.0, 1.0, 0.5]

```
e^2.0 = 7.389
e^1.0 = 2.718
e^0.5 = 1.649

Sum = 7.389 + 2.718 + 1.649 = 11.756

softmax = [7.389/11.756, 2.718/11.756, 1.649/11.756]
        = [0.629, 0.231, 0.140]

Sum check: 0.629 + 0.231 + 0.140 = 1.000 ✓
Prediction: Class 0 (62.9% confidence)
```

**Why exponentials?**
1. Makes everything positive (e^x > 0 always)
2. Amplifies differences between large and small values
3. Mathematically derived from extending sigmoid to multiple classes

## 4e. One-Hot Encoding — Teaching the Network What "Correct" Means

For multi-class, convert integer labels to binary vectors:

```
Label 0 → [1, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Label 3 → [0, 0, 0, 1, 0, 0, 0, 0, 0, 0]
Label 7 → [0, 0, 0, 0, 0, 0, 0, 1, 0, 0]
```

**Why?** The number 7 is not "more than" 3. Digits aren't ordinal. One-hot treats each class as equally different from every other class.

```python
# NumPy trick:
Y_onehot = np.eye(10)[Y]    # Identity matrix indexed by labels
```

## 4f. Categorical Cross-Entropy — Multi-Class Loss Function

```
CCE = -(1/m) × Σ Σ y_true × log(y_predicted)
```

Only the correct class contributes (one-hot has one 1, rest 0s).

**Why log?** Heavily penalizes confident wrong predictions:
- -log(0.9) = 0.105 (confident and correct → small loss)
- -log(0.1) = 2.303 (confident and WRONG → huge loss)

### Output Layer Decision Table (Exam-Critical)

| Task | Output Neurons | Activation | Loss |
|:---|:---|:---|:---|
| Binary classification | 1 | Sigmoid | Binary Cross-Entropy |
| Multi-class (N classes) | N | Softmax | Categorical Cross-Entropy |
| Regression | 1 | None (linear) | MSE |


## Table of Contents### Part I: Concept and Mental Model| # | Section | What You Will Learn ||---|---------|---------------------|| 1 | The Architect's Mental Model — Why CNNs Exist | Why flattening images fails and what CNNs do differently || 2 | The Analogy — A Security Team with Magnifying Glasses | How filters scan images the way a security team inspects documents || 3 | Hello World — Your First CNN | A minimal CNN that classifies MNIST digits || 4 | Why Does This Work? | Local connectivity + parameter sharing — the two principles |### Part II: The Convolution Operation| # | Section | What You Will Learn ||---|---------|---------------------|| 5 | What Is Convolution? | Element-wise multiply, sum, slide — the core operation || 6 | Stride and Padding | How filter movement and border handling affect output size || 7 | The Output Shape Formula | O = (N + 2P - F) / S + 1 — with worked examples || 8 | Multiple Filters and Depth | How K filters produce K feature maps (the depth dimension) || 9 | 1x1 Convolution — The Dimension Controller | Changing depth without changing spatial dimensions |### Part III: Pooling, Flattening, and the Full Pipeline| # | Section | What You Will Learn ||---|---------|---------------------|| 10 | Pooling — Shrinking Feature Maps | Max pooling, average pooling, and why zero parameters || 11 | Flattening and FC Layers | Bridging from 3D feature maps to 1D classification || 12 | The Complete CNN Pipeline | End-to-end architecture with parameter counts || 13 | Weight Counting — The Exam Skill | How to count parameters in any CNN by hand |### Part IV: Build Something Real| # | Section | What You Will Learn ||---|---------|---------------------|| 14 | CNN on MNIST — PyTorch | Build, train, and evaluate a CNN from scratch || 15 | CNN on CIFAR-10 — With Regularization | Dropout, batch norm, data augmentation in a CNN || 16 | MLP vs CNN Comparison | Same dataset, different architectures — see why CNNs win || 17 | Deliberate Mistakes — Learn by Breaking | Common CNN bugs and how to diagnose them |### Part V: Landscape and Production| # | Section | What You Will Learn ||---|---------|---------------------|| 18 | Famous CNN Architectures | LeNet → AlexNet → VGG → GoogLeNet → ResNet — what each discovered || 19 | Transfer Learning Preview | Using pretrained models for your own tasks || 20 | Real-World CNN Applications | Medical imaging, self-driving cars, manufacturing, agriculture, OCR || 21 | Architect's Decision Checklist | When to use CNNs, how many layers, filter sizes, regularization || 22 | Self-Check and Exam Practice | Conceptual questions, calculation problems, design questions |

---## Part I: Concept and Mental Model

## 1. The Architect's Mental Model — Why CNNs ExistBefore building a CNN, understand the problem it solves.### The Problem with Fully Connected Networks for ImagesIn the previous notebook, we flattened MNIST images from 28x28 into 784-dimensional vectors. Each pixel became an independent input feature. This works for simple digits, but creates three serious problems:**Problem 1: Explosion of parameters.**A modest 224x224 color image has 150,528 pixels (224 x 224 x 3 channels). Connect that to a hidden layer of just 512 neurons:| Approach | Parameters (one layer) | What Happens ||:---|:---|:---|| Fully Connected | 150,528 x 512 = **77 million** | Overfits, slow, needs massive GPU memory || CNN (3x3 filter, 32 filters) | (3 x 3 x 3 + 1) x 32 = **896** | Fast, generalizes, runs on a laptop |That is an **86,000x reduction** in parameters for a single layer.**Problem 2: Spatial structure destroyed.**Pixel (0,0) is next to pixel (0,1) in the image — they form part of an edge together. But in a flattened vector, that relationship is invisible. The network must learn from scratch that pixel 0 and pixel 1 are neighbors. For every possible pair. For every possible pattern. That is astronomically wasteful.**Problem 3: No translation invariance.**A fully connected network that learns to recognize a dog in the center of an image cannot recognize the same dog in the corner. It would need to learn "dog at position (100,100)" and "dog at position (10,10)" as completely separate patterns. CNNs apply the same filter everywhere — learn once, detect anywhere.### The CNN Solution: Two Principles| Principle | What It Means | Analogy ||:---|:---|:---|| **Local Connectivity** | Each neuron only looks at a small patch of the input, not the whole image | Reading one paragraph at a time instead of the whole book simultaneously || **Parameter Sharing** | The same filter weights are reused at every spatial position | One security guard who walks the entire building with the same checklist, instead of hiring a different guard for each room |Together, these produce: fewer parameters, preserved spatial structure, and translation invariance.

## 2. The Analogy — A Security Team with Magnifying GlassesImagine you are the head of security for a large art museum. You need to inspect every painting for forgeries.**The Fully Connected Approach (bad):**You hire one expert per square inch of every painting. Each expert only knows about their specific square inch. You need millions of experts, and none of them can detect a pattern that spans two inches.**The CNN Approach (good):**You hire a small team of specialists. Each specialist has a magnifying glass (filter) trained to detect one specific pattern:- **Specialist 1:** Detects brushstroke edges (edge detector)- **Specialist 2:** Detects color gradients (gradient detector)- **Specialist 3:** Detects circular shapes (shape detector)Each specialist walks the entire painting, sliding their magnifying glass from left to right, top to bottom. At each position, they report: "how strongly does my pattern appear here?" The result is a **feature map** — a heat map showing where that pattern exists in the painting.```Painting     →  [Edge Specialist]  →  Edge Map (where are the edges?)             →  [Color Specialist] →  Color Map (where are the gradients?)             →  [Shape Specialist] →  Shape Map (where are the circles?)```This is exactly what a convolutional layer does. The filter IS the specialist. Sliding across the image IS the convolution operation. The output IS the feature map.### Why This Works- **Fewer specialists than square inches** → fewer parameters than FC- **Same specialist covers the whole painting** → parameter sharing- **Each specialist only looks at a local region** → local connectivity- **A forgery is detected regardless of where it appears** → translation invariance

## 3. Hello World — Your First CNNLet's build a minimal CNN and see it work before diving into the theory.

In [ ]:
import torchimport torch.nn as nnimport torch.nn.functional as Ffrom torchvision import datasets, transformsfrom torch.utils.data import DataLoader# === DATA: Load MNIST, normalize to [0,1] ===transform = transforms.Compose([transforms.ToTensor()])train_data = datasets.MNIST(root='./data', train=True, download=True, transform=transform)test_data = datasets.MNIST(root='./data', train=False, download=True, transform=transform)train_loader = DataLoader(train_data, batch_size=64, shuffle=True)test_loader = DataLoader(test_data, batch_size=64, shuffle=False)# === MODEL: A minimal CNN ===class SimpleCNN(nn.Module):    """Minimal CNN for MNIST.         Architecture: Conv(1→16, 3x3) → ReLU → Pool(2x2) → Conv(16→32, 3x3) → ReLU → Pool(2x2) → Flatten → FC(800→10)        WHY this architecture?    - Conv layers preserve spatial structure (unlike flattening to 784)    - Pool layers shrink feature maps (28→14→5, reducing parameters)    - FC layer at the end maps features to 10 digit classes    - ReLU adds non-linearity (edges + shapes = objects)    """    def __init__(self):        super().__init__()        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, padding=1)   # 1 channel in, 16 filters out, 3x3        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)  # 16 channels in, 32 filters out        self.pool = nn.MaxPool2d(2, 2)                             # 2x2 max pooling, stride 2        self.fc = nn.Linear(32 * 7 * 7, 10)                       # Flatten → 10 classes    def forward(self, x):        # x shape: [batch, 1, 28, 28]        x = self.pool(F.relu(self.conv1(x)))   # → [batch, 16, 14, 14]        x = self.pool(F.relu(self.conv2(x)))   # → [batch, 32, 7, 7]        x = x.flatten(start_dim=1)              # → [batch, 1568]        x = self.fc(x)                          # → [batch, 10]        return xmodel = SimpleCNN()print(model)print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}")# Compare: A fully connected network on 784 inputs with same hidden sizes would have ~400,000+ parameters# This CNN has ~16,000 — and it will perform BETTER

In [ ]:
# === TRAIN: The same 5-step heartbeat from the previous notebook ===device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')model = SimpleCNN().to(device)optimizer = torch.optim.Adam(model.parameters(), lr=0.001)criterion = nn.CrossEntropyLoss()# Train for 3 epochs — enough to see it learnfor epoch in range(3):    model.train()    total_loss = 0    correct = 0    total = 0        for images, labels in train_loader:        images, labels = images.to(device), labels.to(device)                optimizer.zero_grad()                    # 1. Zero gradients        outputs = model(images)                  # 2. Forward pass        loss = criterion(outputs, labels)        # 3. Compute loss        loss.backward()                          # 4. Backward pass        optimizer.step()                         # 5. Update weights                total_loss += loss.item()        _, predicted = outputs.max(1)        correct += (predicted == labels).sum().item()        total += labels.size(0)        train_acc = 100 * correct / total        # Evaluate on test set    model.eval()    test_correct = 0    test_total = 0    with torch.no_grad():        for images, labels in test_loader:            images, labels = images.to(device), labels.to(device)            outputs = model(images)            _, predicted = outputs.max(1)            test_correct += (predicted == labels).sum().item()            test_total += labels.size(0)        test_acc = 100 * test_correct / test_total    print(f"Epoch {epoch+1}: Loss={total_loss/len(train_loader):.4f}, Train Acc={train_acc:.1f}%, Test Acc={test_acc:.1f}%")# You should see 98%+ test accuracy in just 3 epochs# Compare: The MLP from the previous notebook needed more epochs for similar accuracy

## 4. Why Does This Work?The CNN above achieved 98%+ accuracy with ~16,000 parameters. The MLP from the previous notebook used ~400,000+ parameters for similar results. Why?### Local ConnectivityEach Conv2d neuron only looks at a 3x3 patch (9 pixels), not all 784 pixels. This means:- 9 connections per neuron instead of 784 (87x fewer)- The neuron learns "what does this local neighborhood look like?" not "what does the entire image look like?"- Neighboring pixels form edges, curves, and textures — exactly what the 3x3 patch captures### Parameter SharingThe same 3x3 filter slides across the entire 28x28 image. One filter = 9 weights + 1 bias = 10 parameters, reused at every position. A fully connected equivalent would need unique weights for every pixel-to-neuron connection.**The math:**- CNN Conv layer: 16 filters x (3x3x1 + 1) = **160 parameters** → produces 16 feature maps- FC equivalent: 784 inputs x 16 outputs = **12,544 parameters** → same number of outputsSame computational result, 78x fewer parameters.### The Feature HierarchyLayer 1 filters learn to detect **edges** (horizontal, vertical, diagonal). Layer 2 filters combine edges into **shapes** (curves, corners, loops). The FC layer maps shapes to **digits** (loop = 0 or 8, vertical line = 1, etc.).This mirrors how the human visual cortex works (Hubel & Wiesel, 1958 — discovered neurons in cats that respond to specific edge orientations).```Raw Pixels → [Conv1: Edge detectors] → [Conv2: Shape detectors] → [FC: Digit classifier]   28x28         "vertical edge"           "loop shape"              "this is a 0"```

---## Part II: The Convolution Operation

## 5. What Is Convolution?### The Mental ModelA convolution is a **pattern-matching scan**. You have a small pattern (the filter) and a large image. You slide the pattern across every position of the image and ask: "how much does this patch match my pattern?"The operation at each position:1. **Overlay** the filter on top of the image patch2. **Multiply** each filter value by the corresponding pixel value (element-wise)3. **Sum** all the products into one number4. **That sum** goes into the output feature map at the corresponding position5. **Slide** the filter to the next position and repeatThis is identical to the weighted sum (W^T x + b) that every neuron computes — the filter values ARE the weights. The image patch IS the input. The result IS the net input before activation.### Worked Example: 5x5 Image, 3x3 Filter, Stride 1```Input (5x5):              Filter (3x3):[1, 0, 1, 0, 1]          [1, 0, 1][0, 1, 0, 1, 0]          [0, 1, 0][1, 0, 1, 0, 1]          [1, 0, 1][0, 1, 0, 1, 0][1, 0, 1, 0, 1]Position (0,0): overlay filter on top-left 3x3 patch:  1*1 + 0*0 + 1*1 + 0*0 + 1*1 + 0*0 + 1*1 + 0*0 + 1*1 = 5Position (0,1): slide one pixel right:  0*1 + 1*0 + 0*1 + 1*0 + 0*1 + 1*0 + 0*1 + 1*0 + 0*1 = 0Output (3x3):[5, 0, 5][0, 5, 0][5, 0, 5]```The output shows WHERE the pattern (a checkerboard-like pattern) is strongest in the image.### In Code

In [ ]:
import numpy as npdef convolution_2d(image, kernel, stride=1):    """Manual 2D convolution — understand what Conv2d does internally.        Args:        image: 2D numpy array (height x width)        kernel: 2D numpy array (filter_h x filter_w)        stride: how many pixels to move between positions        Returns:        2D numpy array — the feature map (activation map)    """    img_h, img_w = image.shape    k_h, k_w = kernel.shape        # Output size formula: O = (N - F) / S + 1    out_h = (img_h - k_h) // stride + 1    out_w = (img_w - k_w) // stride + 1        output = np.zeros((out_h, out_w))        for i in range(out_h):        for j in range(out_w):            # Extract the patch under the filter            patch = image[i*stride : i*stride + k_h,                          j*stride : j*stride + k_w]            # Element-wise multiply and sum            output[i, j] = np.sum(patch * kernel)        return output# === Test it ===image = np.array([    [1, 0, 1, 0, 1],    [0, 1, 0, 1, 0],    [1, 0, 1, 0, 1],    [0, 1, 0, 1, 0],    [1, 0, 1, 0, 1]], dtype=float)kernel = np.array([    [1, 0, 1],    [0, 1, 0],    [1, 0, 1]], dtype=float)result = convolution_2d(image, kernel)print("Input (5x5):")print(image)print("\nKernel (3x3):")print(kernel)print(f"\nOutput ({result.shape[0]}x{result.shape[1]}):")print(result)print(f"\nOutput shape formula: ({image.shape[0]} - {kernel.shape[0]}) / 1 + 1 = {result.shape[0]}")

## 6. Stride and Padding### Stride — How Far the Filter Moves| Stride | Effect | Output Size (7x7 input, 3x3 filter) | When to Use ||:---|:---|:---|:---|| **S=1** | Visits every position, maximum detail | (7-3)/1 + 1 = **5x5** | Default — captures everything || **S=2** | Skips every other position, halves output | (7-3)/2 + 1 = **3x3** | Downsampling (alternative to pooling) || **S=3** | Aggressive skip, loses detail | (7-3)/3 + 1 = **2x2** | Rare — too aggressive for most tasks |### Padding — Preserving DimensionsWithout padding, every convolution layer shrinks the image. After 10 layers of 3x3 convolutions with no padding, a 32x32 image becomes 12x12. You lose 20 pixels of border information.**"Valid" padding (P=0):** No padding. Output shrinks. Edge pixels get less attention.```Input: [0, 1, 2, 3, 4]     Filter: [1, 1, 1]     Output: [3, 6, 9]  (3 values, lost 2)```**"Same" padding (P=1 for 3x3):** Add zeros around the border. Output = input size.```Input: [0, 0, 1, 2, 3, 4, 0]     Filter: [1, 1, 1]     Output: [1, 3, 6, 9, 7]  (5 values, preserved!)         ↑ pad               pad ↑```**Formula for "same" padding:** P = (F - 1) / 2- 3x3 filter: P = 1- 5x5 filter: P = 2- 7x7 filter: P = 3**This is why CNN filters are almost always odd-sized** — even-sized filters can't produce symmetric padding.

## 7. The Output Shape Formula — The Most Important Formula for Exams$$O = \frac{N + 2P - F}{S} + 1$$Where:- **O** = output height or width- **N** = input height or width- **P** = padding- **F** = filter size- **S** = stride### For 3D (with depth/channels):- Output height = (H + 2P - F) / S + 1- Output width = (W + 2P - F) / S + 1- **Output depth = K** (number of filters) — NOT the input depth!The filter must match the input depth (a 3x3 filter on RGB input is actually 3x3x3), but the output depth is determined by how many filters you use.### Worked Examples

In [ ]:
def conv_output_shape(input_size, filter_size, stride=1, padding=0, num_filters=1):    """Calculate output dimensions of a convolution layer.        This is the formula you MUST know for exams.    """    h_in, w_in = input_size if isinstance(input_size, tuple) else (input_size, input_size)        h_out = (h_in + 2 * padding - filter_size) // stride + 1    w_out = (w_in + 2 * padding - filter_size) // stride + 1        if h_out <= 0 or w_out <= 0:        return "INVALID — filter doesn't fit!"        return (h_out, w_out, num_filters)# === Exam-style practice problems ===print("=== Output Shape Calculations ===")print()examples = [    {"input": (7, 7),   "filter": 3, "stride": 1, "padding": 0, "filters": 1,  "desc": "Basic: 7x7, 3x3 filter, stride 1, no padding"},    {"input": (32, 32), "filter": 3, "stride": 1, "padding": 1, "filters": 1,  "desc": "Same padding: 32x32, 3x3, P=1"},    {"input": (32, 32), "filter": 5, "stride": 1, "padding": 2, "filters": 10, "desc": "Same padding: 32x32, 5x5, P=2, 10 filters"},    {"input": (64, 64), "filter": 7, "stride": 2, "padding": 3, "filters": 32, "desc": "Stride 2: 64x64, 7x7, P=3, 32 filters"},    {"input": (28, 28), "filter": 5, "stride": 1, "padding": 0, "filters": 6,  "desc": "LeNet Conv1: 28x28, 5x5, no padding, 6 filters"},]for ex in examples:    result = conv_output_shape(ex["input"], ex["filter"], ex["stride"], ex["padding"], ex["filters"])    print(f"{ex['desc']}")    print(f"  Input: {ex['input']}, Filter: {ex['filter']}x{ex['filter']}, Stride: {ex['stride']}, Padding: {ex['padding']}, Filters: {ex['filters']}")    print(f"  Output: {result}")    print(f"  Formula: ({ex['input'][0]} + 2*{ex['padding']} - {ex['filter']}) / {ex['stride']} + 1 = {result[0] if isinstance(result, tuple) else result}")    print()

## 8. Multiple Filters and Depth### The Mental ModelOne filter detects one type of pattern. But images contain many patterns — horizontal edges, vertical edges, curves, textures, colors. So we use **multiple filters**, each learning to detect a different feature.**One filter = one specialist with one magnifying glass.****K filters = a team of K specialists, each looking for different things.**The outputs of all K filters are **stacked** to form the output volume:```Input: 32x32x3 (RGB image)Apply K=16 filters, each 3x3x3, stride 1, padding 1Output: 32x32x16 (16 feature maps stacked together)```The **depth of the output = number of filters**. This is a design choice:- More filters = more features detected = more expressive = more parameters- Common pattern: 32 → 64 → 128 → 256 (doubles with each layer as spatial dimensions halve)### Why Depth Grows As Spatial Dimensions Shrink| Layer | Spatial Size | Depth | Total Values | What It Captures ||:---|:---|:---|:---|:---|| Input | 32x32 | 3 | 3,072 | Raw pixels || After Conv1+Pool | 16x16 | 32 | 8,192 | Edges, simple textures || After Conv2+Pool | 8x8 | 64 | 4,096 | Shapes, complex textures || After Conv3+Pool | 4x4 | 128 | 2,048 | Parts of objects || After Conv4+Pool | 2x2 | 256 | 1,024 | High-level object features |Spatial dimensions shrink (less "where"), depth grows (more "what"). By the end, the network knows a lot about *what* features are present but less about *exactly where* — which is fine for classification.

## 9. 1x1 Convolution — The Dimension Controller### The Mental ModelA 1x1 convolution sounds useless — how can a single-pixel filter detect spatial patterns? It can't. **That's not its job.**A 1x1 convolution operates across **channels**, not across space. Think of it as a **department consolidator**: if your company has 256 departments each sending reports for every location, the 1x1 conv reads all 256 reports at each location and writes K summary notes.- Spatial dimensions: **unchanged** (1x1 filter at stride 1 visits every pixel once)- Depth: **completely controlled** by number of filters K### Why It Matters: Computational SavingsWithout 1x1 conv: Input 28x28x256 → 3x3 Conv with 256 filters → **~120M multiplications**With 1x1 bottleneck: Input 28x28x256 → 1x1 Conv (16 filters) → 3x3 Conv → **~12.4M multiplications****That's a 10x reduction** by squeezing the depth before the expensive spatial convolution.### Where 1x1 Convolutions Are Used- **GoogLeNet/Inception (2014):** 7M parameters (vs VGG's 138M) — thanks to 1x1 bottlenecks- **ResNet (2015):** Bottleneck blocks: 1x1 → 3x3 → 1x1- **Nearly all modern architectures** use 1x1 convolutions for depth management

In [ ]:
# === 1x1 Convolution Examples ===print("=== 1x1 Convolution Output Shapes ===")print()examples_1x1 = [    {"input": (7, 7, 128),  "filters": 1,   "desc": "Compress 128 channels → 1"},    {"input": (7, 7, 128),  "filters": 16,  "desc": "Compress 128 channels → 16"},    {"input": (28, 28, 256),"filters": 64,  "desc": "Compress 256 channels → 64"},    {"input": (28, 28, 16), "filters": 256, "desc": "Expand 16 channels → 256"},]for ex in examples_1x1:    h, w, d = ex["input"]    k = ex["filters"]    params = (1 * 1 * d + 1) * k  # (F*F*D_in + bias) * K    print(f"{ex['desc']}")    print(f"  Input: {h}x{w}x{d}")    print(f"  1x1 Conv with {k} filters")    print(f"  Output: {h}x{w}x{k}")    print(f"  Parameters: (1*1*{d} + 1) * {k} = {params}")    print()

---## Part III: Pooling, Flattening, and the Full Pipeline

## 10. Pooling — Shrinking Feature Maps### The Mental ModelAfter convolution extracts features, pooling **compresses** the feature maps. It summarizes local regions so the network focuses on "was this feature present?" rather than "where exactly was it?"**Max Pooling** (most common): Takes the maximum value in each region.- "Was a strong edge detected anywhere in this 2x2 area? Yes (keep the max). I don't care exactly which pixel."**Average Pooling:** Takes the mean. Smoother, less aggressive.### Key Properties| Property | Pooling | Convolution ||:---|:---|:---|| Learnable parameters | **0** (fixed operation) | Yes (filter weights + biases) || Changes depth? | **No** | **Yes** (depth = # filters) || Reduces spatial size? | Yes | Only without "same" padding || Most common setting | 2x2, stride 2 (halves dimensions) | 3x3, stride 1 |### Why Zero Parameters?Pooling is a fixed function — there's nothing to learn. "Take the max of these 4 numbers" doesn't require weights. This is unusual — every other layer in the network has learnable parameters.### Pooling Output FormulaSame formula as convolution: O = (N - F) / S + 1With 2x2 pooling, stride 2: O = (N - 2) / 2 + 1 = N/2A 28x28 feature map becomes 14x14. A 14x14 becomes 7x7.

## 13. Weight Counting — The Exam Skill### The Formula**Per convolution layer:** Parameters = (F x F x D_in + 1) x KWhere:- F = filter height/width- D_in = input depth (channels)- +1 = bias per filter- K = number of filters### Worked Example: Full CNN Pipeline```Input: 32x32x3 (CIFAR-10 color image)Layer 1: Conv2d(3→6, 5x5, stride=1, no padding) + ReLU  Output shape: (32-5)/1 + 1 = 28x28x6  Parameters: (5*5*3 + 1) * 6 = 76 * 6 = 456Layer 2: MaxPool(2x2, stride=2)  Output shape: 14x14x6  Parameters: 0 (pooling has no learnable weights!)Layer 3: Conv2d(6→16, 5x5, stride=1, no padding) + ReLU  Output shape: (14-5)/1 + 1 = 10x10x16  Parameters: (5*5*6 + 1) * 16 = 151 * 16 = 2,416Layer 4: MaxPool(2x2, stride=2)  Output shape: 5x5x16  Parameters: 0Layer 5: Flatten  Output shape: 5*5*16 = 400  Parameters: 0Layer 6: FC(400→120) + ReLU  Parameters: (400 + 1) * 120 = 48,120Layer 7: FC(120→84) + ReLU  Parameters: (120 + 1) * 84 = 10,164Layer 8: FC(84→10)  Parameters: (84 + 1) * 10 = 850TOTAL: 456 + 0 + 2,416 + 0 + 0 + 48,120 + 10,164 + 850 = 62,006 parameters```**Notice:** The FC layers have way more parameters than the conv layers (59,134 vs 2,872). This is why modern architectures minimize FC layers and use Global Average Pooling instead.

In [ ]:
def count_cnn_parameters(layers):    """Count parameters in a CNN architecture — exam practice tool.        Each layer is a dict: {'type': 'conv'/'pool'/'flatten'/'fc', ...}    """    total = 0    print(f"{'Layer':<35} {'Output Shape':<20} {'Parameters':>12}")    print("-" * 70)        for layer in layers:        if layer['type'] == 'conv':            f = layer['filter']            d_in = layer['d_in']            k = layer['filters']            params = (f * f * d_in + 1) * k            h_out = (layer['h_in'] + 2*layer.get('padding',0) - f) // layer.get('stride',1) + 1            shape = f"{h_out}x{h_out}x{k}"            print(f"Conv2d({d_in}→{k}, {f}x{f}){'':<15} {shape:<20} {params:>12,}")        elif layer['type'] == 'pool':            h_out = layer['h_in'] // 2            shape = f"{h_out}x{h_out}x{layer['d']}"            params = 0            print(f"MaxPool(2x2){'':<23} {shape:<20} {params:>12,}")        elif layer['type'] == 'flatten':            size = layer['size']            shape = str(size)            params = 0            print(f"Flatten{'':<28} {shape:<20} {params:>12,}")        elif layer['type'] == 'fc':            n_in = layer['in']            n_out = layer['out']            params = (n_in + 1) * n_out            shape = str(n_out)            print(f"FC({n_in}→{n_out}){'':<22} {shape:<20} {params:>12,}")                total += params        print("-" * 70)    print(f"{'TOTAL':<55} {total:>12,}")    return total# === LeNet-5 style architecture ===print("=== LeNet-5 on CIFAR-10 (32x32x3) ===\n")lenet_layers = [    {'type': 'conv', 'h_in': 32, 'filter': 5, 'd_in': 3,  'filters': 6},    {'type': 'pool', 'h_in': 28, 'd': 6},    {'type': 'conv', 'h_in': 14, 'filter': 5, 'd_in': 6,  'filters': 16},    {'type': 'pool', 'h_in': 10, 'd': 16},    {'type': 'flatten', 'size': 400},    {'type': 'fc', 'in': 400, 'out': 120},    {'type': 'fc', 'in': 120, 'out': 84},    {'type': 'fc', 'in': 84,  'out': 10},]count_cnn_parameters(lenet_layers)

---

## 10b. Pooling — Worked Example by Hand

### Max Pooling: 4x4 Feature Map, 2x2 Pool, Stride 2

```
Input feature map (4x4):          Apply 2x2 Max Pooling, Stride 2:

[6, 2, 3, 1]                     Top-left 2x2:  max(6,2,1,4) = 6
[1, 4, 7, 2]                     Top-right 2x2: max(3,1,7,2) = 7
[5, 3, 0, 8]                     Bot-left 2x2:  max(5,3,2,1) = 5
[2, 1, 4, 3]                     Bot-right 2x2: max(0,8,4,3) = 8

Output (2x2):
[6, 7]
[5, 8]
```

**What happened:**
- 4x4 → 2x2 (halved both dimensions)
- Depth unchanged (if input was 4x4x16, output is 2x2x16)
- Zero parameters learned
- The strongest activation in each 2x2 region survived

### Average Pooling (same input):

```
Top-left:  avg(6,2,1,4) = 3.25        Output (2x2):
Top-right: avg(3,1,7,2) = 3.25        [3.25, 3.25]
Bot-left:  avg(5,3,2,1) = 2.75        [2.75, 3.75]
Bot-right: avg(0,8,4,3) = 3.75
```

**Max pooling preserves the strongest signal. Average pooling gives a smoother summary.**


## 10c. Padding Calculation — Worked Problems

### Problem 1: "What padding gives same output?"

**Given:** Input 7x7, Filter 3x3, Stride 1. What P makes output = 7x7?

**Solve:** O = (N + 2P - F) / S + 1
- 7 = (7 + 2P - 3) / 1 + 1
- 7 = 4 + 2P + 1
- 7 = 5 + 2P
- 2P = 2
- **P = 1**

**Verify:** (7 + 2(1) - 3) / 1 + 1 = (7+2-3)/1 + 1 = 6/1 + 1 = 7 ✓

### Problem 2: "What padding gives same output with 5x5 filter?"

**Given:** Input 32x32, Filter 5x5, Stride 1. What P makes output = 32x32?

**Solve:**
- 32 = (32 + 2P - 5) / 1 + 1
- 32 = 27 + 2P + 1
- 32 = 28 + 2P
- 2P = 4
- **P = 2**

**Shortcut:** For "same" padding with stride 1: **P = (F - 1) / 2**
- 3x3 filter: P = (3-1)/2 = **1**
- 5x5 filter: P = (5-1)/2 = **2**
- 7x7 filter: P = (7-1)/2 = **3**

### Problem 3: "Does this filter fit?"

**Given:** Input 5x5, Filter 4x4, Stride 2, Padding 0.

**Solve:** O = (5 + 0 - 4) / 2 + 1 = 1/2 + 1 = 1.5

**Answer: INVALID.** Non-integer result means the filter doesn't fit evenly. You'd need padding or a different stride.


## 12b. Full Forward Pass Through a CNN — Trace Every Value

This is the type of problem the professor asked on Exam 1 (for ANNs). Here it is for CNNs.

### Problem: Given this tiny CNN, trace the values through every layer.

**Input image (4x4, 1 channel, grayscale):**
```
[1, 0, 1, 0]
[0, 1, 0, 1]
[1, 0, 1, 0]
[0, 1, 0, 1]
```

**Layer 1: Conv2d(1→1, 2x2, stride=1, no padding, bias=0)**
Filter:
```
[1, 0]
[0, 1]
```

**Step 1: Convolution** — slide 2x2 filter, compute dot product at each position:
```
Position (0,0): 1*1 + 0*0 + 0*0 + 1*1 = 2
Position (0,1): 0*1 + 1*0 + 1*0 + 0*1 = 0
Position (0,2): 1*1 + 0*0 + 0*0 + 1*1 = 2
Position (1,0): 0*1 + 1*0 + 1*0 + 0*1 = 0
Position (1,1): 1*1 + 0*0 + 0*0 + 1*1 = 2
Position (1,2): 0*1 + 1*0 + 1*0 + 0*1 = 0
Position (2,0): 1*1 + 0*0 + 0*0 + 1*1 = 2
Position (2,1): 0*1 + 1*0 + 1*0 + 0*1 = 0
Position (2,2): 1*1 + 0*0 + 0*0 + 1*1 = 2

Conv output (3x3):
[2, 0, 2]
[0, 2, 0]
[2, 0, 2]
```
Shape check: (4 - 2)/1 + 1 = 3 ✓

**Step 2: ReLU activation**
```
ReLU([2, 0, 2], [0, 2, 0], [2, 0, 2]) = [2, 0, 2], [0, 2, 0], [2, 0, 2]
```
(All values ≥ 0, so no change. If any were negative, they'd become 0.)

**Step 3: MaxPool(2x2, stride=2)**

Wait — 3x3 with 2x2 pool, stride 2: O = (3-2)/2 + 1 = 1.5 → doesn't fit evenly!

With stride 2 on 3x3, we can only do one full 2x2 window starting at (0,0):
```
max(2, 0, 0, 2) = 2

Pool output (1x1): [2]
```
(In practice, frameworks handle this with floor/ceil. For exams, use architectures that divide evenly.)

**Step 4: Flatten**
```
[2] → vector of length 1
```

**Step 5: FC(1→2) with weights W=[0.5, -0.3], bias=[0.1, 0.2]**
```
Output neuron 1: 2 * 0.5 + 0.1 = 1.1
Output neuron 2: 2 * (-0.3) + 0.2 = -0.4
```

**Step 6: Softmax**
```
softmax([1.1, -0.4]) = [e^1.1 / (e^1.1 + e^-0.4), e^-0.4 / (e^1.1 + e^-0.4)]
                      = [3.004 / (3.004 + 0.670), 0.670 / 3.674]
                      = [0.818, 0.182]
```

**Prediction: Class 0 (81.8% confidence)**

### Summary of shapes through the network:
```
Input:     4x4x1
Conv(2x2): 3x3x1     (456→parameters: (2*2*1+0)*1 = 4 weights, 0 bias = 4)
ReLU:      3x3x1     (0 parameters)
Pool(2x2): 1x1x1     (0 parameters)
Flatten:   1         (0 parameters)
FC(1→2):   2         ((1+1)*2 = 4 parameters)
Softmax:   2         (0 parameters)
Total: 8 parameters
```


In [ ]:
# === VERIFY the forward pass above with PyTorch ===
import torch
import torch.nn as nn
import torch.nn.functional as F

# Create the exact input
x = torch.tensor([[[[1., 0., 1., 0.],
                     [0., 1., 0., 1.],
                     [1., 0., 1., 0.],
                     [0., 1., 0., 1.]]]])  # Shape: [1, 1, 4, 4]

# Create conv layer with our exact filter
conv = nn.Conv2d(1, 1, kernel_size=2, bias=False)
with torch.no_grad():
    conv.weight.copy_(torch.tensor([[[[1., 0.], [0., 1.]]]]))

# Trace through
print("Input (4x4):")
print(x.squeeze().numpy())

conv_out = conv(x)
print(f"\nAfter Conv2d(2x2): shape {list(conv_out.shape)}")
print(conv_out.squeeze().detach().numpy())

relu_out = F.relu(conv_out)
print(f"\nAfter ReLU: (no change since all ≥ 0)")
print(relu_out.squeeze().detach().numpy())

pool_out = F.max_pool2d(relu_out, 2)
print(f"\nAfter MaxPool(2x2): shape {list(pool_out.shape)}")
print(pool_out.squeeze().detach().numpy())

flat = pool_out.flatten(start_dim=1)
print(f"\nAfter Flatten: shape {list(flat.shape)}, value = {flat.item():.1f}")

# FC layer
fc = nn.Linear(1, 2)
with torch.no_grad():
    fc.weight.copy_(torch.tensor([[0.5], [-0.3]]))
    fc.bias.copy_(torch.tensor([0.1, 0.2]))

fc_out = fc(flat)
print(f"\nAfter FC(1→2): {fc_out.detach().numpy()}")

softmax_out = F.softmax(fc_out, dim=1)
print(f"After Softmax: {softmax_out.detach().numpy()}")
print(f"\nPrediction: Class {softmax_out.argmax().item()} ({softmax_out.max().item()*100:.1f}% confidence)")


---## Companion Exercises — Hands-On NotebooksThese Pluralsight notebooks contain **runnable code** that brings the concepts above to life. Work through them after reading the corresponding section.**Location:** `~/Downloads/image-classification-pytorch/` and `~/Downloads/foundations-pytorch/`| After Reading... | Run This Notebook | What You Will See ||:---|:---|:---|| **Section 5** (Convolution) | `04_UnderstandingConvolutionFilters.ipynb` | Apply edge detection, sharpening, and line detection filters to a real street photo. See the feature maps with your own eyes. || **Section 6** (Stride/Padding) | `08_KernelSize.ipynb` | Train the same CNN with different kernel sizes on CIFAR-10. See how kernel size affects accuracy. || **Section 8** (Multiple Filters) | `06_ActivationLayers.ipynb` | Build a CNN on CIFAR-10 with batch normalization. Compare different activation functions. || **Section 10** (Pooling) | `07_PoolingLayers.ipynb` | Full CIFAR-10 CNN training with MaxPool2d. Achieves 69.66% test accuracy. See loss curves. || **Section 13** (Weight Counting) | `05_CNNs_HyperparameterTuning.ipynb` | Tune hidden layer sizes (16→32 vs 32→64) on MNIST. See parameter count changes. || **Section 19** (Transfer Learning) | `09_TransferLearning.ipynb` | Load pretrained ResNet-18, freeze backbone, fine-tune on CIFAR-10. 74.89% accuracy in 1 epoch. || **Section 20** (Real-World) | `01_ImagePreprocessingTechniques.ipynb` | Resize, crop, denoise, flip, rotate real images (birds, giraffes). Data augmentation in practice. |### Additional PyTorch Fundamentals (from `~/Downloads/foundations-pytorch/`)| Topic | Notebook | What It Demonstrates ||:---|:---|:---|| Autograd basics | `demo5-AutogradIntro.ipynb` | requires_grad, backward(), gradient accumulation, torch.no_grad() || Linear regression | `demo7-LinearModelUsingAutograd.ipynb` | End-to-end training with manual weight updates + matplotlib visualization || PyTorch vs TensorFlow | `demo9-PytorchComputationGraph.ipynb` | Dynamic vs static computation graphs |### How to Use These1. **Read the concept** in this notebook first (build the mental model)2. **Run the companion notebook** (see it work with real data)3. **Come back here** for the "why does this work?" and exam practice questions4. **Break something** in the companion notebook — change a kernel size, remove a pooling layer, set stride=5 — and observe what happens

---## 22. Self-Check and Exam Practice### Conceptual Questions (No Code)1. **Why do CNNs use local connectivity instead of full connectivity?**2. **What is weight sharing and why does it reduce parameters?**3. **What happens if you use no padding with 10 layers of 3x3 convolutions on a 32x32 image?** (Calculate the final size)4. **Pooling has zero learnable parameters. Why?**5. **Why does the depth (channels) typically increase while spatial dimensions decrease through a CNN?**6. **When would you use 1x1 convolution? What problem does it solve?**7. **What is the difference between max pooling and average pooling?**8. **Why are CNN filters almost always odd-sized (3x3, 5x5, 7x7)?**### Calculation Problems (Show Your Work)9. **Output shape:** Input 64x64x3, Filter 5x5, Stride 2, Padding 1, 32 filters. What is the output shape?10. **Output shape with pooling:** Input 28x28x1 → Conv(1→16, 3x3, P=1) → MaxPool(2x2) → Conv(16→32, 3x3, P=1) → MaxPool(2x2). What is the final shape?11. **Parameter count:** How many parameters in a Conv2d(64→128, 3x3) layer?12. **Parameter count:** How many parameters in the full network from Q10 if followed by Flatten → FC(?) → FC(10)?13. **1x1 conv:** Input 14x14x512. Apply 1x1 Conv with 64 filters. What is the output shape and how many parameters?### Answers<details><summary>Click to reveal answers</summary>9. O = (64 + 2*1 - 5) / 2 + 1 = 61/2 + 1 = 31. Output: **31x31x32**10. Conv1: 28x28x16, Pool1: 14x14x16, Conv2: 14x14x32, Pool2: **7x7x32**11. (3*3*64 + 1) * 128 = 577 * 128 = **73,856**12. Flatten: 7*7*32 = 1568. Conv1: (3*3*1+1)*16=160. Conv2: (3*3*16+1)*32=4,640. FC1: (1568+1)*128=200,832 (if 128 hidden). FC2: (128+1)*10=1,290. Total depends on FC hidden size.13. Output: **14x14x64**. Parameters: (1*1*512 + 1) * 64 = 513 * 64 = **32,832**</details>